In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split

# Generating Mock Data

In [2]:
def generate_mock_data(num_samples=1000):

    np.random.seed(42) 
    
    # Splitting the dataset 50/50 between the two classes "Real Experts" and "Frauds"
    half = num_samples // 2
    
 
    real_experts = pd.DataFrame({
        'Perplexity_Score': np.clip(np.random.normal(0.7, 0.15, half), 0, 1),
        'Burstiness_Score': np.clip(np.random.normal(0.65, 0.2, half), 0, 1),
        'Skill_Velocity_Index': np.clip(np.random.normal(0.3, 0.1, half), 0, 1),
        'Semantic_Delta': np.clip(np.random.normal(0.25, 0.1, half), 0, 1),
        'Digital_Footprint_Density': np.clip(np.random.normal(0.8, 0.15, half), 0, 1),
        'Is_Fraud': 0
    })
    

    frauds = pd.DataFrame({
        'Perplexity_Score': np.clip(np.random.normal(0.2, 0.1, half), 0, 1),
        'Burstiness_Score': np.clip(np.random.normal(0.15, 0.1, half), 0, 1),
        'Skill_Velocity_Index': np.clip(np.random.normal(0.85, 0.1, half), 0, 1),
        'Semantic_Delta': np.clip(np.random.normal(0.8, 0.1, half), 0, 1),
        'Digital_Footprint_Density': np.clip(np.random.normal(0.1, 0.1, half), 0, 1),
        'Is_Fraud': 1
    })
    
   
    dataset = pd.concat([real_experts, frauds]).sample(frac=1).reset_index(drop=True)
    
    return dataset

mock_dataset = generate_mock_data(1000)

# Save to CSV for the RL environment to consume
mock_dataset.to_csv("expert_verification_data.csv", index=False)
print(mock_dataset.head())

   Perplexity_Score  Burstiness_Score  Skill_Velocity_Index  Semantic_Delta  \
0          0.746636          0.525372              0.182989        0.185377   
1          0.346418          0.227516              0.766805        0.763688   
2          0.837310          0.558128              0.403028        0.267644   
3          0.435544          0.529203              0.438009        0.372693   
4          0.168647          0.265390              1.000000        0.878359   

   Digital_Footprint_Density  Is_Fraud  
0                   0.958990         0  
1                   0.000000         1  
2                   0.723779         0  
3                   0.823279         0  
4                   0.307507         1  


In [3]:
class ExpertVerificationEnv:
    def __init__(self, dataframe):
        self.df = dataframe
        self.current_step = 0
        self.max_steps = len(dataframe)
        
    def reset(self):
        self.current_step = 0
        return self._get_observation()
        
    def _get_observation(self):
        row = self.df.iloc[self.current_step]
        # Extracting the State Space features
        obs = np.array([
            row['Perplexity_Score'],
            row['Burstiness_Score'],
            row['Skill_Velocity_Index'],
            row['Semantic_Delta'],
            row['Digital_Footprint_Density']
        ], dtype=np.float32)
        return obs
        
    def step(self, action_prob):
        p = np.clip(action_prob, 0.0, 1.0)
        y_true = self.df.iloc[self.current_step]['Is_Fraud']
        
        # Calculating Reward based on the created asymmetrical logic
        reward = self._calculate_reward(p, y_true)
        
        self.current_step += 1
        done = self.current_step >= self.max_steps
        next_obs = self._get_observation() if not done else None
        
        return next_obs, reward, done, {"actual_label": y_true, "predicted_prob": p}
        
    def _calculate_reward(self, p, y):
        if y == 1:
            # For When a Candidate IS a "Fraud". 
            # If p=1.0 (True Positive), Reward = +10. 
            # If p=0.0 (False Negative), Reward = -20.
            return (30 * p) - 20
        else:
            # For When a Candidate is a "REAL Expert" (y=0).
            # If p=0.0 (True Negative), Reward = +2.
            # If p=1.0 (False Positive), Reward = -5.
            return (7 * (1 - p)) - 5

# --- Testing the Environment and simulating a response from the untrained agent ---
env = ExpertVerificationEnv(mock_dataset)
initial_state = env.reset()

print("Initial State Space (Features):", initial_state)

simulated_action = 0.85 
next_state, reward, done, info = env.step(simulated_action)

print(f"Action (Predicted Prob): {simulated_action}")
print(f"Ground Truth: {info['actual_label']}")
print(f"Calculated Reward: {reward}")

Initial State Space (Features): [0.74663615 0.5253719  0.1829887  0.18537727 0.95899045]
Action (Predicted Prob): 0.85
Ground Truth: 0.0
Calculated Reward: -3.9499999999999997


In [4]:
# Splitting the data (90% Train, 10% Test)
train_df, test_df = train_test_split(mock_dataset, test_size=0.10, random_state=42)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Training Candidates: {len(train_df)}")
print(f"Testing Candidates: {len(test_df)}")

Training Candidates: 900
Testing Candidates: 100


In [5]:
# Initializing Two Separate Environments for the Data split
train_env = ExpertVerificationEnv(train_df)
test_env = ExpertVerificationEnv(test_df)

In [6]:
# Agent and Optimizer Setup
class FraudDetectionAgent(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(5, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
        
    def forward(self, state):
        return self.network(state)

agent = FraudDetectionAgent()
optimizer = optim.Adam(agent.parameters(), lr=0.01)

In [7]:
# Training the Agent 
epochs = 50
print("\n--- Starting Training Phase ---")

# Setting the model to training mode
agent.train() 

for epoch in range(epochs):
    state = train_env.reset()
    total_train_reward = 0
    epoch_loss = 0
    
    for step in range(train_env.max_steps):
        state_tensor = torch.FloatTensor(state)
        action_prob = agent(state_tensor)
        
        next_state, reward, done, info = train_env.step(action_prob.item())
        total_train_reward += reward
        y_true = info['actual_label']
        
        
        if y_true == 1:
            loss = -(30 * action_prob - 20)
        else:
            loss = -(7 * (1.0 - action_prob) - 5)
            
        epoch_loss += loss
        state = next_state
        if done: break
            
    epoch_loss = epoch_loss / train_env.max_steps
    optimizer.zero_grad()
    epoch_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | Train Reward: {total_train_reward:.2f}")


--- Starting Training Phase ---
Epoch 01/50 | Train Reward: -2782.17
Epoch 10/50 | Train Reward: -1128.28
Epoch 20/50 | Train Reward: 1032.92
Epoch 30/50 | Train Reward: 3061.27
Epoch 40/50 | Train Reward: 4439.23
Epoch 50/50 | Train Reward: 5012.31


In [8]:
# Evaluating model with the 10% validation split
print("\n--- Starting Testing Phase (Unseen Data) ---")

# Setting the model to evaluation mode
agent.eval() 
state = test_env.reset()
total_test_reward = 0

# Disabling gradient tracking 
with torch.no_grad(): 
    for step in range(test_env.max_steps):
        state_tensor = torch.FloatTensor(state)
        action_prob = agent(state_tensor)
        
        next_state, reward, done, info = test_env.step(action_prob.item())
        total_test_reward += reward
        
        state = next_state
        if done: break


max_possible_test_reward = 0
for _, row in test_df.iterrows():
    if row['Is_Fraud'] == 1:
        max_possible_test_reward += 10 # True Positive
    else:
        max_possible_test_reward += 2  # True Negative

print(f"Total Test Reward: {total_test_reward:.2f} / {max_possible_test_reward}")
print(f"Test Set Performance: {(total_test_reward / max_possible_test_reward) * 100:.2f}% of Theoretical Maximum")


--- Starting Testing Phase (Unseen Data) ---
Total Test Reward: 524.50 / 560
Test Set Performance: 93.66% of Theoretical Maximum


In [9]:
# Save the model
torch.save(agent.state_dict(), 'fraud_detection_agent.pth')
print("Model saved to fraud_detection_agent.pth")

Model saved to fraud_detection_agent.pth
